In [1]:
import pandas as pd
import os

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

df = pd.read_csv('../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv')
en = df[df['language'] == 'en'].dropna(subset=['body', 'answer']).reset_index(drop=True)
print(en.shape)

c:\Users\User\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(16335, 16)


In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = model.encode(
    en['body'].tolist(), 
    show_progress_bar=True, 
    batch_size=64
)
print("Embeddings shape:", embeddings.shape)

Batches: 100%|██████████| 256/256 [04:12<00:00,  1.01it/s]


Embeddings shape: (16335, 384)


In [3]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

print("Total vectors in index:", index.ntotal)

os.makedirs('../src/rag', exist_ok=True)
faiss.write_index(index, '../src/rag/ticket_index.faiss')

# Also save the corresponding metadata (body + answer + ticket info) so we can look up results later
en[['subject', 'body', 'answer', 'type', 'queue', 'priority']].to_csv('../src/rag/ticket_metadata.csv', index=False)

print("Saved index and metadata")

Total vectors in index: 16335
Saved index and metadata


In [4]:
def search_similar_tickets(query_text, k=3):
    query_embedding = model.encode([query_text])
    distances, indices = index.search(np.array(query_embedding).astype('float32'), k)
    
    results = en.iloc[indices[0]][['subject', 'body', 'answer', 'priority']]
    return results

# Test with a made-up new ticket
test_query = "My internet keeps disconnecting during video calls and I've already restarted my router"
results = search_similar_tickets(test_query)
for i, row in results.iterrows():
    print("Subject:", row['subject'])
    print("Body:", row['body'][:200])
    print("Answer:", row['answer'][:200])
    print("Priority:", row['priority'])
    print("---")

Subject: Immediate Help Needed for Video Conference System Connectivity Issues
Body: Dear Customer Support,\n\nI am writing to urgently request assistance regarding ongoing connection problems with our video conferencing system. Despite attempts to resolve the matter, the system conti
Answer: Thank you for reaching out. To assist you better, could you specify the version of the video conferencing software you are using, the error messages encountered, and details about your operating syste
Priority: medium
---
Subject: Immediate Help Needed for Video Conferencing System Connectivity Issues
Body: Dear Customer Support,\n\nI am reaching out to request immediate assistance regarding ongoing connectivity problems with our video conferencing system. Despite our attempts to resolve the matter, the 
Answer: Thank you for contacting us. To assist you more effectively, could you specify which version of the video conferencing software you are using, any error messages you have encountered, and 